<a href="https://colab.research.google.com/github/snaval29/Aakaar-AI/blob/main/AakaarAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
## 1. Model: MusicGen-Small (`facebook/musicgen-small`)
* **Type:** State-of-the-art text-to-instrumental music model.
* **Hardware:** Runs efficiently on an NVIDIA T4 GPU (takes ~10-15s per track).
* **Auth:** No Hugging Face token or external API keys needed.
```

```markdown
## 2. System Workflow
1. **Environment Setup:** Install requirements and check for GPU.
2. **Model Loading:** Load processor and model into memory.
3. **Music Generation:** Use prompt processing and inference to generate audio.
4. **Gradio UI:** Deploy a web interface to generate and download audio tracks.
```

```markdown
## 3. Installation
Install the necessary libraries for model processing, GPU acceleration, audio compilation, and UI deployment.
```

In [ ]:
# Install required libraries
!pip install -q transformers==4.30.2 accelerate soundfile gradio

# Transformers: Provides the MusicGen model and processor.
# Accelerate: Helps with efficient GPU usage, especially for larger models.
# Soundfile: Used for saving the generated audio arrays into a .wav file.
# Gradio: Enables quick creation of a web-based user interface for our model.

print("All required libraries installed successfully!")

```markdown
## 4. Hardware Check
Verify CUDA availability to leverage GPU acceleration.
```

In [9]:
# This cell now consolidated into the unified setup cell below.
print("Hardware, model, and interface setup have been moved to the main application cell.")

Hardware, model, and interface setup have been moved to the main application cell.


```markdown
## 5. Load Model
Load tokenizer/processor and model architecture to device memory.
```

In [10]:
# This cell now consolidated into the unified setup cell below.
print("Model loading logic integrated into the unified runner cell.")

Model loading logic integrated into the unified runner cell.


```markdown
## 6. Generation Pipeline
Process user inputs, configure dynamic tokens based on requested duration, and output compiled audio.
```

In [11]:
# This cell now consolidated into the unified setup cell below.
print("Core generator integrated into the unified runner cell.")

Core generator integrated into the unified runner cell.


```markdown
## 7. Interactive Gradio Interface
Consolidated interface runner cell featuring model loading, audio synthesis, and live playbacks.
```

In [ ]:
import os
import re
import uuid
import gc
import torch
import numpy as np
import soundfile as sf
import gradio as gr

from transformers import AutoProcessor, MusicgenForConditionalGeneration

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODEL_NAME = "facebook/musicgen-small"
OUTPUT_DIR = os.path.join(os.getcwd(), "generated_audio")
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU mode: generation may be slow.")

# ==========================================
# 2. PROMPT PROCESSING
# ==========================================

WORD_CORRECTIONS = {
    "voilen": "violin",
    "voilin": "violin",
    "violien": "violin",
    "loofy": "lo-fi",
    "lofi": "lo-fi",
    "raininy": "rainy",
    "soothng": "soothing",
    "relxing": "relaxing",
    "paino": "piano",
    "drumm": "drums",
}

def process_prompt(user_prompt):
    if not isinstance(user_prompt, str) or not user_prompt.strip():
        raise ValueError("Please enter a music description.")

    prompt = re.sub(r"\s+", " ", user_prompt.strip())

    for wrong, correct in WORD_CORRECTIONS.items():
        prompt = re.sub(
            rf"(?<!\w){re.escape(wrong)}(?!\w)",
            correct,
            prompt,
            flags=re.IGNORECASE
        )

    # Preserve the user's idea; do not force a genre template.
    prompt = (
        "Instrumental music, "
        + prompt
        + ", coherent musical arrangement, high quality audio"
    )

    return prompt

# ==========================================
# 3. LOAD MODEL ONCE
# ==========================================

print("Loading MusicGen model...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

SAMPLE_RATE = model.config.audio_encoder.sampling_rate

print("MusicGen loaded successfully.")
print("Sample rate:", SAMPLE_RATE)

# ==========================================
# 4. MUSIC GENERATION
# ==========================================

def generate_music(user_prompt, duration):
    try:
        duration = int(duration)

        if not 5 <= duration <= 15:
            raise ValueError("Duration must be between 5 and 15 seconds.")

        final_prompt = process_prompt(user_prompt)

        print("\nOriginal prompt:", user_prompt)
        print("Processed prompt:", final_prompt)

        inputs = processor(
            text=[final_prompt],
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # MusicGen uses approximately 50 generated tokens per second.
        max_new_tokens = duration * 50

        with torch.inference_mode():
            audio_values = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                guidance_scale=3.0
            )

        print("Audio tensor shape:", tuple(audio_values.shape))

        if audio_values.ndim != 3 or audio_values.shape[-1] == 0:
            raise RuntimeError("The model returned an empty audio tensor.")

        # Extract first track: [channels, samples]
        audio = audio_values[0].detach().float().cpu().numpy()
        audio = np.squeeze(audio)

        if audio.ndim == 1:
            pass
        elif audio.ndim == 2:
            audio = audio.T
        else:
            raise RuntimeError("Unexpected audio tensor dimensions.")

        if audio.size == 0 or not np.isfinite(audio).all():
            raise RuntimeError("Audio contains no valid samples.")

        peak = float(np.max(np.abs(audio)))
        rms = float(np.sqrt(np.mean(np.square(audio))))

        print("Peak amplitude:", peak)
        print("RMS amplitude:", rms)

        if peak < 1e-5 or rms < 1e-7:
            raise RuntimeError(
                "Generated audio is nearly silent. Try generating again."
            )

        filename = f"music_{uuid.uuid4().hex[:10]}.wav"
        output_path = os.path.join(OUTPUT_DIR, filename)

        sf.write(
            output_path,
            audio,
            SAMPLE_RATE,
            subtype="PCM_16"
        )

        # Verify the saved audio file.
        saved_audio, saved_sr = sf.read(output_path)
        file_size = os.path.getsize(output_path)
        saved_duration = len(saved_audio) / saved_sr

        if len(saved_audio) == 0 or file_size < 1000:
            raise RuntimeError("Saved audio file is empty or invalid.")

        print("Saved:", output_path)
        print("File size:", file_size, "bytes")
        print("Duration:", round(saved_duration, 2), "seconds")

        return (
            output_path,
            f"Generation successful. Prompt used: {final_prompt}"
        )

    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return None, "GPU memory exhausted. Try a 5-second generation."

    except Exception as exc:
        print("Generation error:", repr(exc))
        return None, f"Generation failed: {exc}"

# ==========================================
# 5. GRADIO INTERFACE
# ==========================================

def gradio_wrapper(prompt, duration):
    audio_path, status = generate_music(prompt, duration)
    return status, audio_path

with gr.Blocks(title="Text-to-Music Generator") as demo:
    gr.Markdown("# AI Text-to-Music Generator")
    gr.Markdown(
        "Describe any music idea in your own words. "
        "You are not restricted to predefined genres."
    )

    prompt_input = gr.Textbox(
        label="Describe your music",
        placeholder=(
            "Rainy night, piano and violin, "
            "with a smooth lo-fi atmosphere..."
        ),
        lines=3
    )

    duration_input = gr.Slider(
        minimum=5,
        maximum=15,
        value=5,
        step=1,
        label="Duration (seconds)"
    )

    generate_button = gr.Button(
        "Generate Music",
        variant="primary"
    )

    status_output = gr.Textbox(
        label="Status",
        interactive=False
    )

    audio_output = gr.Audio(
        label="Generated Music",
        type="filepath"
    )

    generate_button.click(
        fn=gradio_wrapper,
        inputs=[prompt_input, duration_input],
        outputs=[status_output, audio_output]
    )

demo.launch(share=True, debug=True)

Using device: cuda
GPU: Tesla T4
Loading MusicGen model...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicGen loaded successfully.
Sample rate: 32000
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e4a014ad19d04cf199.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


```markdown
## 8. Quick Testing Steps
1. Run the installation cell, followed by the integrated generator cell.
2. Open the public or inline Gradio web UI.
3. Input prompt ideas (e.g., *'lo-fi piano and violin'*) and select duration, then click **Generate**.
```

```markdown
## 9. Troubleshooting & Common Errors
* **GPU Memory (OOM):** Reduce generation duration.
* **Silent Audio:** Try standard instruments or adjust prompts.
* **Build Failures:** Restart runtime and run the installation cell again.
```

```markdown
## 10. Future Milestones
* API integrations for enhanced performance.
* Web frameworks (e.g., FastAPI backend) to isolate components.
* Pre/post-processing filters for mastering output tracks.
```